# CAD results of iFlip on sentiment and news classification

In [ ]:
import os
import gc
import torch
import pandas as pd
from collections import Counter

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score

# =========================
# CONFIG
# =========================
base_model = "bert-base-uncased"
data_dir = "results_llama"
output_root = "bert_finetuned_models_llama"
report_path = "bert_finetuned_models_llama/eval_report.csv"
os.makedirs(output_root, exist_ok=True)

MAX_EVAL_SAMPLES = 1500
MAX_LENGTH = 256
STRIDE = 128
SEED = 3407

# canonical label space for our heads
label_maps = {
    "imdb": {"pos": 1, "neg": 0, "positive": 1, "negative": 0},
    "agnews": {"world": 0, "sports": 1, "business": 2, "sci/tech": 3},
    "snli": {"entailment": 0, "neutral": 1, "contradiction": 2},
}

ood_sets = {
    "imdb": [("amazon_polarity", None), ("sst2", "glue")],
    "agnews": [("bbc-news", "SetFit"), ("20_newsgroups", "SetFit")],
    "snli": [("multi_nli", None), ("anli", None)],
}


def map_label_any(task: str, value):
    s = str(value).strip()
    if s.lstrip("-").isdigit():
        return int(s)
    key = s.lower()
    if key in label_maps[task]:
        return label_maps[task][key]
    print(f"[WARN] Unmapped label '{value}' for task '{task}', defaulting to 0")
    return 0

def prepare_train_dataset_from_csv(csv_path: str, task: str) -> Dataset:
    df = pd.read_csv(csv_path)
    texts = list(df["original_review"]) + list(df["counterfactual"])
    raw_labels = list(df["ground_truth_label"]) + list(df["cf_pred_label"])

    # map strings/numbers -> canonical ids
    mapped = []
    for l in raw_labels:
        s = str(l).strip()
        if s.lstrip("-").isdigit():
            mapped.append(int(s))
        else:
            mapped.append(label_maps[task].get(s.lower(), -999))  # unknown -> invalid

    # keep only labels in [0, num_labels-1]
    num_classes = len(label_maps[task])
    keep_mask = [(0 <= y < num_classes) for y in mapped]
    dropped = len(mapped) - sum(keep_mask)
    if dropped > 0:
        print(f"[{task}] Drop {dropped} training rows with invalid labels (e.g., -1).")

    texts  = [t for t, k in zip(texts, keep_mask) if k]
    labels = [y for y, k in zip(mapped, keep_mask) if k]
    ids    = list(range(len(texts)))

    return Dataset.from_dict({"id": ids, "text": texts, "label": labels})


def tokenize_sliding_batched(batch, tokenizer):
    """Return flat rows: one row per window (no nested lists)."""
    texts = batch["text"]
    ids = batch["id"]
    if "label" in batch:
        labels_in = batch["label"]
    elif "labels" in batch:
        labels_in = batch["labels"]
    else:
        labels_in = [None] * len(texts)

    all_input_ids, all_attn, all_labels, all_sids = [], [], [], []
    for i, text in enumerate(texts):
        tok = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            padding="max_length",
            return_overflowing_tokens=True,
            return_offsets_mapping=False,
        )
        n = len(tok["input_ids"])
        all_input_ids.extend(tok["input_ids"])
        all_attn.extend(tok["attention_mask"])
        all_labels.extend([labels_in[i]] * n)
        all_sids.extend([ids[i]] * n)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attn,
        "labels": all_labels,
        "sample_id": all_sids,
    }

def majority_vote(preds, sample_ids):
    per_id = {}
    for p, sid in zip(preds, sample_ids):
        per_id.setdefault(sid, []).append(p)
    sorted_ids = sorted(per_id.keys())
    final = [Counter(per_id[sid]).most_common(1)[0][0] for sid in sorted_ids]
    return final, sorted_ids

def evaluate_with_voting(trainer: Trainer, tokenized_eval_ds):
    out = trainer.predict(tokenized_eval_ds)
    window_preds = out.predictions.argmax(-1)
    sample_ids = tokenized_eval_ds["sample_id"]

    # pick the first label per sample_id (same label across windows)
    first_label_by_id = {}
    for sid, lab in zip(sample_ids, tokenized_eval_ds["labels"]):
        if sid not in first_label_by_id:
            first_label_by_id[sid] = lab

    final_preds, sorted_ids = majority_vote(window_preds, sample_ids)
    final_labels = [first_label_by_id[sid] for sid in sorted_ids]
    return {
        "accuracy": accuracy_score(final_labels, final_preds),
        "f1_macro": f1_score(final_labels, final_preds, average="macro"),
    }

# ---------- in-domain ----------
def load_in_domain_eval(task: str) -> Dataset:
    if task == "imdb":
        ds = load_dataset("imdb", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        # imdb already has 'text'
        return ds

    if task == "agnews":
        ds = load_dataset("ag_news", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        # ag_news already has 'text'
        return ds

    if task == "snli":
        ds = load_dataset("snli", split="test")
        ds = ds.filter(lambda x: x["label"] != -1)
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        # build single text field consistent with our training
        ds = ds.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        # remap labels to our SNLI order using class names if present
        if isinstance(ds.features["label"], type(ds.features["label"])):
            # ds.features["label"] is ClassLabel; convert via names
            names = ds.features["label"].names
            name_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
            ds = ds.map(lambda ex: {"labels": name_map[names[ex["label"]]]})
        else:
            # fallback: keep ints as-is (assuming same order)
            ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        return ds

    raise ValueError(f"Unknown task {task}")

# ---------- OOD remapping helpers ----------

def get_label_names(ds, label_col="labels"):

    feat = ds.features.get(label_col, None)
    # Case 1: ClassLabel
    if feat is not None and hasattr(feat, "names") and isinstance(feat.names, (list, tuple)):
        return [str(n).lower() for n in feat.names]

    # Case 2: derive from auxiliary text column
    cand_cols = [c for c in ds.column_names if c.lower() in ("label_text", "labels_text", "category", "topic", "class_name")]
    if cand_cols:
        tcol = cand_cols[0]
        sample = ds.select(range(min(2000, len(ds))))  # small subset
        lab_list = sample[label_col]
        txt_list = [str(t).lower() for t in sample[tcol]]
        buckets = {}
        for lab, txt in zip(lab_list, txt_list):
            buckets.setdefault(lab, Counter()).update([txt])
        max_lab = max(lab_list) if len(lab_list) else -1
        names = [None] * (max_lab + 1)
        for lab, counter in buckets.items():
            names[lab] = counter.most_common(1)[0][0]
        return [n if n is not None else str(i) for i, n in enumerate(names)]

    # Case 3: unknown
    return None


def remap_bbc_to_agnews_ids(ds):

    names = get_label_names(ds, "labels")
    if names is None:
        fallback_map = {0: 2, 1: 0, 2: 0, 3: 1, 4: 3}
        return ds.map(lambda ex: {"labels": fallback_map.get(ex["labels"], 0)})

    idx2ag = {}
    for i, n in enumerate(names):
        if "sport" in n:
            idx2ag[i] = 1
        elif "tech" in n or "sci" in n or "science" in n or "technology" in n:
            idx2ag[i] = 3
        elif "business" in n or "money" in n or "finance" in n or "econom" in n:
            idx2ag[i] = 2
        elif "politic" in n or "world" in n or "intl" in n or "international" in n or "uk" in n:
            idx2ag[i] = 0
        elif "entertain" in n or "culture" in n or "arts" in n:
            idx2ag[i] = 0
        else:
            idx2ag[i] = 0
    return ds.map(lambda ex: {"labels": idx2ag.get(ex["labels"], 0)})


def remap_20ng_to_agnews_ids(ds):

    names = get_label_names(ds, "labels")
    if names is None:
        return ds.map(lambda ex: {"labels": 0})

    def to_ag(name: str) -> int:
        n = name.lower()
        if n.startswith("rec.sport"):
            return 1
        if n.startswith("comp.") or n.startswith("sci."):
            return 3
        if n == "misc.forsale":
            return 2
        if n.startswith("talk.politics") or n.startswith("soc."):
            return 0
        return 0

    idx2ag = {i: to_ag(n) for i, n in enumerate(names)}
    return ds.map(lambda ex: {"labels": idx2ag.get(ex["labels"], 0)})

def remap_snli_style(ds):
    # for MNLI/ANLI -> align to our SNLI id order
    names = ds.features["labels"].names if "labels" in ds.features else None
    if names is None:
        return ds
    name_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    def _map_lab(ex):
        return {"labels": name_map[names[ex["labels"]]]}
    return ds.map(_map_lab)

# ---------- OOD ----------
def load_ood_eval(task: str):
    sets = []

    if task == "imdb":
        # Amazon Polarity
        ds = load_dataset("amazon_polarity", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.map(lambda ex: {"text": (ex.get("title", "") + ". " + ex.get("content", "")).strip()})
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("amazon_polarity", ds))

        # SST-2
        ds = load_dataset("glue", "sst2", split="validation")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.rename_column("sentence", "text")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("sst2", ds))

    elif task == "agnews":
        # BBC News (5 → mapped to 4)
        ds = load_dataset("SetFit/bbc-news", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        ds = remap_bbc_to_agnews_ids(ds)
        sets.append(("bbc-news", ds))

        # 20 Newsgroups (20 → mapped to 4)
        ds = load_dataset("SetFit/20_newsgroups", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        ds = remap_20ng_to_agnews_ids(ds)
        sets.append(("20_newsgroups", ds))

    elif task == "snli":
        # -------- MNLI (validation) --------
        try:
            ds = load_dataset("multi_nli", split="validation_matched")
        except Exception:
            ds = load_dataset("multi_nli", split="validation_mismatched")
        ds = ds.filter(lambda x: x["label"] != -1)
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("mnli", ds))


        ds_r1 = load_dataset("anli", "plain_text", split="dev_r1")
        ds_r2 = load_dataset("anli", "plain_text", split="dev_r2")
        ds_r3 = load_dataset("anli", "plain_text", split="dev_r3")
        ds_anli = concatenate_datasets([ds_r1, ds_r2, ds_r3])


        ds_anli = ds_anli.filter(lambda x: x["label"] != -1)
        ds_anli = ds_anli.select(range(min(MAX_EVAL_SAMPLES, len(ds_anli))))


        ds_anli = ds_anli.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        ds_anli = ds_anli.rename_column("label", "labels")
        ds_anli = ds_anli.add_column("id", list(range(len(ds_anli))))
        sets.append(("anli", ds_anli))

    else:
        raise ValueError(f"Unknown task {task}")

    return sets


def main():
    results_summary = []
    tokenizer = AutoTokenizer.from_pretrained(base_model)

    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".csv"):
            continue

        # Task detection from filename
        if "_imdb_" in fname:
            task = "imdb"
        elif "_agnews_" in fname:
            task = "agnews"
        elif "_snli_" in fname:
            task = "snli"
        else:
            continue
        
        if task == "snli":
            continue

        model_name = f"bert_{fname.replace('.csv', '')}"
        model_out_dir = os.path.join(output_root, model_name)
        os.makedirs(model_out_dir, exist_ok=True)
        print(f"\n=== Fine-tuning {model_name} ({task}) ===")

        # -------- Train set (CSV) --------
        train_ds = prepare_train_dataset_from_csv(os.path.join(data_dir, fname), task)
        train_ds = train_ds.map(
            lambda batch: tokenize_sliding_batched(batch, tokenizer),
            batched=True,
            remove_columns=train_ds.column_names,
            desc="Tokenizing train (sliding window)",
        )

        # -------- In-domain eval --------
        in_domain = load_in_domain_eval(task)
        in_domain = in_domain.map(
            lambda batch: tokenize_sliding_batched(batch, tokenizer),
            batched=True,
            remove_columns=in_domain.column_names,
            desc="Tokenizing in-domain (sliding window)",
        )

        # -------- Model & Training --------
        model = AutoModelForSequenceClassification.from_pretrained(
            base_model, num_labels=len(label_maps[task])
        )

        training_args = TrainingArguments(
            output_dir=model_out_dir,
            save_strategy="epoch",          # no evaluation_strategy (manual eval after training)
            learning_rate=2e-5,
            per_device_train_batch_size=128,
            per_device_eval_batch_size=128,
            num_train_epochs=5,
            weight_decay=0.01,
            logging_steps=100,
            save_total_limit=1,
            report_to="none",
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            tokenizer=tokenizer,  # deprecation warning ok; v5 uses processing_class
        )

        trainer.train()

        # -------- In-domain eval (majority vote) --------
        in_metrics = evaluate_with_voting(trainer, in_domain)
        results_summary.append({
            "model": model_name, "task": task, "dataset": "in_domain",
            "accuracy": in_metrics["accuracy"], "f1_macro": in_metrics["f1_macro"],
        })
        print(f"[In-domain] acc={in_metrics['accuracy']:.4f}, f1={in_metrics['f1_macro']:.4f}")

        # -------- OOD evals --------
        for ood_name, ood_ds in load_ood_eval(task):
            ood_ds = ood_ds.map(
                lambda batch: tokenize_sliding_batched(batch, tokenizer),
                batched=True,
                remove_columns=ood_ds.column_names,
                desc=f"Tokenizing OOD {ood_name} (sliding window)",
            )
            ood_metrics = evaluate_with_voting(trainer, ood_ds)
            results_summary.append({
                "model": model_name, "task": task, "dataset": ood_name,
                "accuracy": ood_metrics["accuracy"], "f1_macro": ood_metrics["f1_macro"],
            })
            print(f"[OOD:{ood_name}] acc={ood_metrics['accuracy']:.4f}, f1={ood_metrics['f1_macro']:.4f}")

        # Save model and free VRAM
        trainer.save_model(model_out_dir)
        del model, trainer
        torch.cuda.empty_cache()
        gc.collect()

    pd.DataFrame(results_summary).to_csv(report_path, index=False)
    print(f"\nEvaluation report saved to {report_path}")

if __name__ == "__main__":
    main()


2025-10-27 21:18:52.204942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761571132.229818    1035 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761571132.237274    1035 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761571132.256879    1035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761571132.256904    1035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761571132.256907    1035 computation_placer.cc:177] computation placer alr


=== Fine-tuning bert_results_conf_agnews_None_counterfactuals_20250705_062145 (agnews) ===


Tokenizing train (sliding window):   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1035/2686215236.py:428: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1035/2686215236.py:428: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


[In-domain] acc=0.9153, f1=0.4779


[OOD:amazon_polarity] acc=0.8827, f1=0.8826


[OOD:sst2] acc=0.8280, f1=0.8269

Evaluation report saved to /root/autodl-tmp/imdb/augmentation/bert_finetuned_models_llama/eval_report.csv


# CAD results of iFlip on NLI

In [ ]:
import os
import gc
import torch
import pandas as pd
from collections import Counter

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score


base_model = "bert-base-uncased"
data_dir = "results_llama"
output_root = "bert_finetuned_models_llama"
report_path = "bert_finetuned_models_llama/eval_report.csv"
os.makedirs(output_root, exist_ok=True)

MAX_EVAL_SAMPLES = 1500
MAX_LENGTH = 256
STRIDE = 128
SEED = 3407

# canonical label space for our heads
label_maps = {
    "imdb": {"pos": 1, "neg": 0, "positive": 1, "negative": 0},
    "agnews": {"world": 0, "sports": 1, "business": 2, "sci/tech": 3},
    "snli": {"entailment": 0, "neutral": 1, "contradiction": 2},
}

ood_sets = {
    "imdb": [("amazon_polarity", None), ("sst2", "glue")],
    "agnews": [("bbc-news", "SetFit"), ("20_newsgroups", "SetFit")],
    "snli": [("multi_nli", None), ("anli", None)],
}

def map_label_any(task: str, value):
    s = str(value).strip()
    if s.lstrip("-").isdigit():
        return int(s)
    key = s.lower()
    if key in label_maps[task]:
        return label_maps[task][key]
    print(f"[WARN] Unmapped label '{value}' for task '{task}', defaulting to 0")
    return 0

def prepare_train_dataset_from_csv(csv_path: str, task: str) -> Dataset:
    df = pd.read_csv(csv_path)
    texts = list(df["original_review"]) + list(df["counterfactual"])
    raw_labels = list(df["ground_truth_label"]) + list(df["cf_pred_label"])

    # map strings/numbers -> canonical ids
    mapped = []
    for l in raw_labels:
        s = str(l).strip()
        if s.lstrip("-").isdigit():
            mapped.append(int(s))
        else:
            mapped.append(label_maps[task].get(s.lower(), -999))  # unknown -> invalid

    # keep only labels in [0, num_labels-1]
    num_classes = len(label_maps[task])
    keep_mask = [(0 <= y < num_classes) for y in mapped]
    dropped = len(mapped) - sum(keep_mask)
    if dropped > 0:
        print(f"[{task}] Drop {dropped} training rows with invalid labels (e.g., -1).")

    texts  = [t for t, k in zip(texts, keep_mask) if k]
    labels = [y for y, k in zip(mapped, keep_mask) if k]
    ids    = list(range(len(texts)))

    return Dataset.from_dict({"id": ids, "text": texts, "label": labels})


def tokenize_sliding_batched(batch, tokenizer):
    """Return flat rows: one row per window (no nested lists)."""
    texts = batch["text"]
    ids = batch["id"]
    if "label" in batch:
        labels_in = batch["label"]
    elif "labels" in batch:
        labels_in = batch["labels"]
    else:
        labels_in = [None] * len(texts)

    all_input_ids, all_attn, all_labels, all_sids = [], [], [], []
    for i, text in enumerate(texts):
        tok = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            padding="max_length",
            return_overflowing_tokens=True,
            return_offsets_mapping=False,
        )
        n = len(tok["input_ids"])
        all_input_ids.extend(tok["input_ids"])
        all_attn.extend(tok["attention_mask"])
        all_labels.extend([labels_in[i]] * n)
        all_sids.extend([ids[i]] * n)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attn,
        "labels": all_labels,
        "sample_id": all_sids,
    }

def majority_vote(preds, sample_ids):
    per_id = {}
    for p, sid in zip(preds, sample_ids):
        per_id.setdefault(sid, []).append(p)
    sorted_ids = sorted(per_id.keys())
    final = [Counter(per_id[sid]).most_common(1)[0][0] for sid in sorted_ids]
    return final, sorted_ids

def evaluate_with_voting(trainer: Trainer, tokenized_eval_ds):
    out = trainer.predict(tokenized_eval_ds)
    window_preds = out.predictions.argmax(-1)
    sample_ids = tokenized_eval_ds["sample_id"]

    # pick the first label per sample_id (same label across windows)
    first_label_by_id = {}
    for sid, lab in zip(sample_ids, tokenized_eval_ds["labels"]):
        if sid not in first_label_by_id:
            first_label_by_id[sid] = lab

    final_preds, sorted_ids = majority_vote(window_preds, sample_ids)
    final_labels = [first_label_by_id[sid] for sid in sorted_ids]
    return {
        "accuracy": accuracy_score(final_labels, final_preds),
        "f1_macro": f1_score(final_labels, final_preds, average="macro"),
    }

# ---------- in-domain ----------
def load_in_domain_eval(task: str) -> Dataset:
    if task == "imdb":
        ds = load_dataset("imdb", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        # imdb already has 'text'
        return ds

    if task == "agnews":
        ds = load_dataset("ag_news", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        # ag_news already has 'text'
        return ds

    if task == "snli":
        ds = load_dataset("snli", split="test")
        ds = ds.filter(lambda x: x["label"] != -1)
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        # build single text field consistent with our training
        ds = ds.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        # remap labels to our SNLI order using class names if present
        if isinstance(ds.features["label"], type(ds.features["label"])):
            # ds.features["label"] is ClassLabel; convert via names
            names = ds.features["label"].names
            name_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
            ds = ds.map(lambda ex: {"labels": name_map[names[ex["label"]]]})
        else:
            # fallback: keep ints as-is (assuming same order)
            ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        return ds

    raise ValueError(f"Unknown task {task}")

# ---------- OOD remapping helpers ----------

def get_label_names(ds, label_col="labels"):
    """
    Try to recover class names for an integer label column.
    1) If it's a ClassLabel, use .names.
    2) Else, if there's a text column like 'label_text'/'category', derive names
       by taking the most frequent text per id.
    3) Else, return None.
    """
    feat = ds.features.get(label_col, None)
    # Case 1: ClassLabel
    if feat is not None and hasattr(feat, "names") and isinstance(feat.names, (list, tuple)):
        return [str(n).lower() for n in feat.names]

    # Case 2: derive from auxiliary text column
    cand_cols = [c for c in ds.column_names if c.lower() in ("label_text", "labels_text", "category", "topic", "class_name")]
    if cand_cols:
        tcol = cand_cols[0]
        sample = ds.select(range(min(2000, len(ds))))  # small subset
        lab_list = sample[label_col]
        txt_list = [str(t).lower() for t in sample[tcol]]
        buckets = {}
        for lab, txt in zip(lab_list, txt_list):
            buckets.setdefault(lab, Counter()).update([txt])
        max_lab = max(lab_list) if len(lab_list) else -1
        names = [None] * (max_lab + 1)
        for lab, counter in buckets.items():
            names[lab] = counter.most_common(1)[0][0]
        return [n if n is not None else str(i) for i, n in enumerate(names)]

    # Case 3: unknown
    return None


def remap_bbc_to_agnews_ids(ds):

    names = get_label_names(ds, "labels")
    if names is None:
        fallback_map = {0: 2, 1: 0, 2: 0, 3: 1, 4: 3}
        return ds.map(lambda ex: {"labels": fallback_map.get(ex["labels"], 0)})

    idx2ag = {}
    for i, n in enumerate(names):
        if "sport" in n:
            idx2ag[i] = 1
        elif "tech" in n or "sci" in n or "science" in n or "technology" in n:
            idx2ag[i] = 3
        elif "business" in n or "money" in n or "finance" in n or "econom" in n:
            idx2ag[i] = 2
        elif "politic" in n or "world" in n or "intl" in n or "international" in n or "uk" in n:
            idx2ag[i] = 0
        elif "entertain" in n or "culture" in n or "arts" in n:
            idx2ag[i] = 0
        else:
            idx2ag[i] = 0
    return ds.map(lambda ex: {"labels": idx2ag.get(ex["labels"], 0)})


def remap_20ng_to_agnews_ids(ds):

    names = get_label_names(ds, "labels")
    if names is None:
        return ds.map(lambda ex: {"labels": 0})

    def to_ag(name: str) -> int:
        n = name.lower()
        if n.startswith("rec.sport"):
            return 1
        if n.startswith("comp.") or n.startswith("sci."):
            return 3
        if n == "misc.forsale":
            return 2
        if n.startswith("talk.politics") or n.startswith("soc."):
            return 0
        return 0

    idx2ag = {i: to_ag(n) for i, n in enumerate(names)}
    return ds.map(lambda ex: {"labels": idx2ag.get(ex["labels"], 0)})

def remap_snli_style(ds):
    # for MNLI/ANLI -> align to our SNLI id order
    names = ds.features["labels"].names if "labels" in ds.features else None
    if names is None:
        return ds
    name_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    def _map_lab(ex):
        return {"labels": name_map[names[ex["labels"]]]}
    return ds.map(_map_lab)

# ---------- OOD ----------
def load_ood_eval(task: str):
    sets = []

    if task == "imdb":
        # Amazon Polarity
        ds = load_dataset("amazon_polarity", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.map(lambda ex: {"text": (ex.get("title", "") + ". " + ex.get("content", "")).strip()})
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("amazon_polarity", ds))

        # SST-2
        ds = load_dataset("glue", "sst2", split="validation")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.rename_column("sentence", "text")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("sst2", ds))

    elif task == "agnews":
        # BBC News (5 → mapped to 4)
        ds = load_dataset("SetFit/bbc-news", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        ds = remap_bbc_to_agnews_ids(ds)
        sets.append(("bbc-news", ds))

        # 20 Newsgroups (20 → mapped to 4)
        ds = load_dataset("SetFit/20_newsgroups", split="test")
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        ds = remap_20ng_to_agnews_ids(ds)
        sets.append(("20_newsgroups", ds))

    elif task == "snli":
        # -------- MNLI (validation) --------
        try:
            ds = load_dataset("multi_nli", split="validation_matched")
        except Exception:
            ds = load_dataset("multi_nli", split="validation_mismatched")
        ds = ds.filter(lambda x: x["label"] != -1)
        ds = ds.select(range(min(MAX_EVAL_SAMPLES, len(ds))))
        ds = ds.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        ds = ds.rename_column("label", "labels")
        ds = ds.add_column("id", list(range(len(ds))))
        sets.append(("mnli", ds))


        ds_r1 = load_dataset("anli", "plain_text", split="dev_r1")
        ds_r2 = load_dataset("anli", "plain_text", split="dev_r2")
        ds_r3 = load_dataset("anli", "plain_text", split="dev_r3")
        ds_anli = concatenate_datasets([ds_r1, ds_r2, ds_r3])


        ds_anli = ds_anli.filter(lambda x: x["label"] != -1)
        ds_anli = ds_anli.select(range(min(MAX_EVAL_SAMPLES, len(ds_anli))))


        ds_anli = ds_anli.map(lambda ex: {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"})
        ds_anli = ds_anli.rename_column("label", "labels")
        ds_anli = ds_anli.add_column("id", list(range(len(ds_anli))))
        sets.append(("anli", ds_anli))

    else:
        raise ValueError(f"Unknown task {task}")

    return sets


def main():
    results_summary = []
    tokenizer = AutoTokenizer.from_pretrained(base_model)

    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".csv"):
            continue

        # Task detection from filename
        if "_imdb_" in fname:
            task = "imdb"
        elif "_agnews_" in fname:
            task = "agnews"
        elif "_snli_" in fname:
            task = "snli"
        else:
            continue
        
        if task != "snli":
            continue

        model_name = f"bert_{fname.replace('.csv', '')}"
        model_out_dir = os.path.join(output_root, model_name)
        os.makedirs(model_out_dir, exist_ok=True)
        print(f"\n=== Fine-tuning {model_name} ({task}) ===")

        # -------- Train set (CSV) --------
        train_ds = prepare_train_dataset_from_csv(os.path.join(data_dir, fname), task)
        train_ds = train_ds.map(
            lambda batch: tokenize_sliding_batched(batch, tokenizer),
            batched=True,
            remove_columns=train_ds.column_names,
            desc="Tokenizing train (sliding window)",
        )

        # -------- In-domain eval --------
        in_domain = load_in_domain_eval(task)
        in_domain = in_domain.map(
            lambda batch: tokenize_sliding_batched(batch, tokenizer),
            batched=True,
            remove_columns=in_domain.column_names,
            desc="Tokenizing in-domain (sliding window)",
        )

        # -------- Model & Training --------
        model = AutoModelForSequenceClassification.from_pretrained(
            base_model, num_labels=len(label_maps[task])
        )

        training_args = TrainingArguments(
            output_dir=model_out_dir,
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=128,
            per_device_eval_batch_size=128,
            num_train_epochs=5,
            weight_decay=0.01,
            logging_steps=100,
            save_total_limit=1,
            report_to="none",
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            tokenizer=tokenizer,
        )

        trainer.train()

        # -------- In-domain eval (majority vote) --------
        in_metrics = evaluate_with_voting(trainer, in_domain)
        results_summary.append({
            "model": model_name, "task": task, "dataset": "in_domain",
            "accuracy": in_metrics["accuracy"], "f1_macro": in_metrics["f1_macro"],
        })
        print(f"[In-domain] acc={in_metrics['accuracy']:.4f}, f1={in_metrics['f1_macro']:.4f}")

        # -------- OOD evals --------
        for ood_name, ood_ds in load_ood_eval(task):
            ood_ds = ood_ds.map(
                lambda batch: tokenize_sliding_batched(batch, tokenizer),
                batched=True,
                remove_columns=ood_ds.column_names,
                desc=f"Tokenizing OOD {ood_name} (sliding window)",
            )
            ood_metrics = evaluate_with_voting(trainer, ood_ds)
            results_summary.append({
                "model": model_name, "task": task, "dataset": ood_name,
                "accuracy": ood_metrics["accuracy"], "f1_macro": ood_metrics["f1_macro"],
            })
            print(f"[OOD:{ood_name}] acc={ood_metrics['accuracy']:.4f}, f1={ood_metrics['f1_macro']:.4f}")

        # Save model and free VRAM
        trainer.save_model(model_out_dir)
        del model, trainer
        torch.cuda.empty_cache()
        gc.collect()

    pd.DataFrame(results_summary).to_csv(report_path, index=False)
    print(f"\nEvaluation report saved to {report_path}")

if __name__ == "__main__":
    main()


2025-10-27 23:07:24.453412: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761577644.479544     980 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761577644.487490     980 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761577644.508517     980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761577644.508544     980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761577644.508547     980 computation_placer.cc:177] computation placer alr


=== Fine-tuning bert_results_conf_snli_hypothesis_counterfactuals_20250709_025949 (snli) ===
[snli] Drop 15 training rows with invalid labels (e.g., -1).


Tokenizing train (sliding window):   0%|          | 0/985 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_980/2779284427.py:428: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_980/2779284427.py:428: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss


[In-domain] acc=0.3620, f1=0.2864


[OOD:mnli] acc=0.3247, f1=0.2619


[OOD:anli] acc=0.3273, f1=0.1669

Evaluation report saved to /root/autodl-tmp/imdb/augmentation/bert_finetuned_models_llama/eval_report.csv


# CAD results of iFlip (tested on human-annotated CF)

In [ ]:
import os
import gc
import torch
import pandas as pd
from collections import Counter

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score


data_dir = "/root/autodl-tmp/imdb/augmentation/human_annotated_cf"
model_root = "/root/autodl-tmp/imdb/augmentation/bert_finetuned_models_llama"
report_path = os.path.join(model_root, "eval_report_human_cf.csv")

MAX_LENGTH = 256
STRIDE = 128
SEED = 3407

label_maps = {
    "imdb": {"pos": 1, "neg": 0, "positive": 1, "negative": 0},
    "snli": {"entailment": 0, "neutral": 1, "contradiction": 2},
}

def map_label(task: str, value):
    s = str(value).strip()
    if s.lstrip("-").isdigit():
        return int(s)
    key = s.lower()
    if key in label_maps[task]:
        return label_maps[task][key]
    return 0


def load_sentiment_tsv(tsv_path: str) -> Dataset:
    df = pd.read_csv(tsv_path, sep="\t")
    cols = {c.lower(): c for c in df.columns}
    text_col = cols.get("text", "Text")
    lab_col = cols.get("sentiment", "Sentiment")
    df = df[[text_col, lab_col]].dropna()
    labels = [map_label("imdb", v) for v in df[lab_col].tolist()]
    texts = df[text_col].astype(str).tolist()
    ids = list(range(len(texts)))
    return Dataset.from_dict({"id": ids, "text": texts, "label": labels})

def load_nli_tsv(tsv_path: str) -> Dataset:
    df = pd.read_csv(tsv_path, sep="\t")
    cols = {c.lower(): c for c in df.columns}
    s1_col = cols.get("sentence1", "sentence1")
    s2_col = cols.get("sentence2", "sentence2")
    lab_col = cols.get("gold_label", "gold_label")
    df = df[[s1_col, s2_col, lab_col]].dropna()
    labels = [map_label("snli", v) for v in df[lab_col].tolist()]
    texts = [f"Premise: {a}\nHypothesis: {b}" for a, b in zip(df[s1_col].astype(str), df[s2_col].astype(str))]
    ids = list(range(len(texts)))
    return Dataset.from_dict({"id": ids, "text": texts, "label": labels})


def tokenize_sliding_batched(batch, tokenizer):
    texts = batch["text"]
    ids = batch["id"]
    labels_in = batch.get("label", [None] * len(texts))

    all_input_ids, all_attn, all_labels, all_sids = [], [], [], []
    for i, text in enumerate(texts):
        tok = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            padding="max_length",
            return_overflowing_tokens=True,
        )
        n = len(tok["input_ids"])
        all_input_ids.extend(tok["input_ids"])
        all_attn.extend(tok["attention_mask"])
        all_labels.extend([labels_in[i]] * n)
        all_sids.extend([ids[i]] * n)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attn,
        "labels": all_labels,
        "sample_id": all_sids,
    }

def majority_vote(preds, sample_ids):
    per_id = {}
    for p, sid in zip(preds, sample_ids):
        per_id.setdefault(sid, []).append(p)
    sorted_ids = sorted(per_id.keys())
    final = [Counter(per_id[sid]).most_common(1)[0][0] for sid in sorted_ids]
    return final, sorted_ids

def evaluate_with_voting(trainer: Trainer, tokenized_eval_ds):
    out = trainer.predict(tokenized_eval_ds)
    window_preds = out.predictions.argmax(-1)
    sample_ids = tokenized_eval_ds["sample_id"]

    first_label_by_id = {}
    for sid, lab in zip(sample_ids, tokenized_eval_ds["labels"]):
        if sid not in first_label_by_id:
            first_label_by_id[sid] = lab

    final_preds, sorted_ids = majority_vote(window_preds, sample_ids)
    final_labels = [first_label_by_id[sid] for sid in sorted_ids]
    return {
        "accuracy": accuracy_score(final_labels, final_preds),
        "f1_macro": f1_score(final_labels, final_preds, average="macro"),
        "n_samples": len(sorted_ids),
    }


def eval_model(task_name: str, test_ds: Dataset, num_labels: int, model_dir: str):
    if len(test_ds) == 0:
        return {"accuracy": None, "f1_macro": None, "n_samples": 0}

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, num_labels=num_labels)

    eval_tok = test_ds.map(
        lambda batch: tokenize_sliding_batched(batch, tokenizer),
        batched=True,
        remove_columns=test_ds.column_names,
        desc=f"[{task_name}] Tokenizing eval"
    )

    tmp_out = os.path.join(model_dir, "_tmp_eval_human_cf")
    os.makedirs(tmp_out, exist_ok=True)

    args = TrainingArguments(
        output_dir=tmp_out,
        per_device_eval_batch_size=64,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
    metrics = evaluate_with_voting(trainer, eval_tok)

    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()
    return metrics






# =========================
# Main
# =========================
def main():
    results = []

    # Sentiment
    sent_ds = load_sentiment_tsv(os.path.join(data_dir, "sentiment.tsv"))
    for subdir in ["bert_results_conf_imdb_None_counterfactuals_20250702_103518",
                   "bert_results_gradxinput_imdb_None_counterfactuals_20250709_012414",
                   "bert_results_lime_imdb_None_counterfactuals_20250708_210756",
                   "bert_results_lxt_imdb_None_counterfactuals_20250703_090558",
                   "bert_results_nl_imdb_None_counterfactuals_20251003_161624",
                   "bert_results_shap_imdb_None_counterfactuals_20250704_074601"]:
        model_path = os.path.join(model_root, subdir)
        metrics = eval_model("imdb_sentiment", sent_ds, num_labels=4, model_dir=model_path)
        results.append({"model": subdir, "task": "imdb", "dataset": "sentiment.tsv (human CF)", **metrics})

    # SNLI Premise
    prem_ds = load_nli_tsv(os.path.join(data_dir, "revised_premise.tsv"))
    for subdir in ["bert_results_conf_snli_premise_counterfactuals_20250708_042501",
                   "bert_results_gradxinput_snli_premise_counterfactuals_20250708_125642",
                   "bert_results_lime_snli_premise_counterfactuals_20250708_091203",
                   "bert_results_lxt_snli_premise_counterfactuals_20250707_163726",
                   "bert_results_nl_snli_premise_counterfactuals_20251003_161440",
                   "bert_results_shap_snli_premise_counterfactuals_20250708_141801"]:
        model_path = os.path.join(model_root, subdir)
        metrics = eval_model("snli_premise", prem_ds, num_labels=3, model_dir=model_path)
        results.append({"model": subdir, "task": "snli_premise", "dataset": "revised_premise.tsv (human CF)", **metrics})

    # SNLI Hypothesis
    hyp_ds = load_nli_tsv(os.path.join(data_dir, "revised_hypothesis.tsv"))
    for subdir in ["bert_results_conf_snli_hypothesis_counterfactuals_20250709_025949",
                   "bert_results_gradxinput_snli_hypothesis_counterfactuals_20250708_185619",
                   "bert_results_lime_snli_hypothesis_counterfactuals_20250708_153840",
                   "bert_results_lxt_snli_hypothesis_counterfactuals_20250709_102822",
                   "bert_results_nl_snli_hypothesis_counterfactuals_20251003_161600",
                   "bert_results_shap_snli_hypothesis_counterfactuals_20250709_201956"]:
        model_path = os.path.join(model_root, subdir)
        metrics = eval_model("snli_hypothesis", hyp_ds, num_labels=3, model_dir=model_path)
        results.append({"model": subdir, "task": "snli_hypothesis", "dataset": "revised_hypothesis.tsv (human CF)", **metrics})



    pd.DataFrame(results).to_csv(report_path, index=False)
    print("\n=== Human CF Evaluation Summary ===")
    print(pd.DataFrame(results))
    print(f"\nSaved to: {report_path}")

if __name__ == "__main__":
    main()


[imdb_sentiment] Tokenizing eval:   0%|          | 0/3414 [00:00<?, ? examples/s]

/tmp/ipykernel_951/3547011601.py:151: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[imdb_sentiment] Tokenizing eval:   0%|          | 0/3414 [00:00<?, ? examples/s]

/tmp/ipykernel_951/3547011601.py:151: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[imdb_sentiment] Tokenizing eval:   0%|          | 0/3414 [00:00<?, ? examples/s]

...

[snli_hypothesis] Tokenizing eval:   0%|          | 0/3332 [00:00<?, ? examples/s]

/tmp/ipykernel_951/3547011601.py:151: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[snli_hypothesis] Tokenizing eval:   0%|          | 0/3332 [00:00<?, ? examples/s]

/tmp/ipykernel_951/3547011601.py:151: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.



=== Human CF Evaluation Summary ===
                                                model             task  \
0   bert_results_conf_imdb_None_counterfactuals_20...             imdb   
1   bert_results_gradxinput_imdb_None_counterfactu...             imdb   
2   bert_results_lime_imdb_None_counterfactuals_20...             imdb   
3   bert_results_lxt_imdb_None_counterfactuals_202...             imdb   
4   bert_results_nl_imdb_None_counterfactuals_2025...             imdb   
5   bert_results_shap_imdb_None_counterfactuals_20...             imdb   
6   bert_results_conf_snli_premise_counterfactuals...     snli_premise   
7   bert_results_gradxinput_snli_premise_counterfa...     snli_premise   
8   bert_results_lime_snli_premise_counterfactuals...     snli_premise   
9   bert_results_lxt_snli_premise_counterfactuals_...     snli_premise   
10  bert_results_nl_snli_premise_counterfactuals_2...     snli_premise   
11  bert_results_shap_snli_premise_counterfactuals...     snli_premise   
1